In [1]:
import torch
import cv2
import numpy as np

import os
from extraction.images import model_loader
from extraction.video_processing import extract_video_features_compressed_ms

import time

In [2]:
ucf_path = 'data/ucf_mini'
violence_dir = ['Fighting', 'Arrest', 'Abuse', 'Assault']
non_violence_dir = ['Normal']

In [3]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [6]:
# 1. Device Guard Setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading transformer weights into active hardware storage space...")

# 2. Initialize weights ONCE at the top level of the cell
global_model, global_processor, active_code = model_loader(model_code='clip', device=device)
print(f"✅ Transformer successfully cached on: {device.upper()}\n")

⏳ Loading transformer weights into active hardware storage space...
✅ Transformer successfully cached on: MPS



In [9]:
violent_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in violence_dir:
    dir_path = os.path.join(ucf_path, video_dir)
    if not os.path.isdir(dir_path): continue
    
    for video_file in os.listdir(dir_path):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(dir_path, video_file)
        print(f"Processing: {video_path}")
        
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=16
        )
        
        violent_data.append(vector)
        execution_speed = time.time() - start_timer

        print("📊 APPLE SILICON PERFORMANCE AUDIT:")
        print(f"🎬 Video Evaluated: {video_path.split('/')[-1]}")

Processing: data/ucf_mini/Fighting/Fighting036_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting036_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting037_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting037_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting030_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting030_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting005_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting005_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting013_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting013_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting048_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting048_x264.mp4
Processing: data/ucf_mini/Fighting/Fighting022_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting022_x264.mp4
Processing: data/ucf_mini/Arrest/Arrest015_x264.mp4
📊 APPLE SILICON PERFORMANCE AUD

In [10]:
non_violent_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in non_violence_dir:
    dir_path = os.path.join(ucf_path, video_dir)
    if not os.path.isdir(dir_path): continue
    
    for video_file in os.listdir(dir_path):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(dir_path, video_file)
        print(f"Processing: {video_path}")
        
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=16
        )
        
        non_violent_data.append(vector)
        execution_speed = time.time() - start_timer

        print("📊 APPLE SILICON PERFORMANCE AUDIT:")
        print(f"🎬 Video Evaluated: {video_path.split('/')[-1]}")

Processing: data/ucf_mini/Normal/Normal_Videos_606_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_606_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_781_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_781_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_189_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_189_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_941_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_941_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_365_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_365_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_696_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_696_x264.mp4
Processing: data/ucf_mini/Normal/Normal_Videos_010_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_010_x264.mp4


In [20]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [26]:
violent_labels = np.ones(len(violent_data))
non_violent_labels = np.zeros(len(non_violent_data))

violent_data = np.array(violent_data)
non_violent_data = np.array(non_violent_data)

In [27]:
x = np.concat([violent_data, non_violent_data])
y = np.concat([violent_labels, non_violent_labels])

In [29]:
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2)

In [30]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.6667 (When flagged positive, accuracy is 66.67%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8333 (When flagged positive, accuracy is 83.33%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 1.0000 (When flagged positive, accuracy is 100.00%)
Custom Recall Score:    0.8000 (Captured 80.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 1.0000 (When flagged positive, accuracy is 100.00%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8000 (When flagged positive, accuracy is 80.00%)
Cus

In [31]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         1
         1.0       1.00      1.00      1.00         6

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

[[1 0]
 [0 6]]


In [32]:
import joblib

In [33]:
joblib.dump(log_reg, open("model/violence.jobllib", 'wb'))